In [ ]:
from langchain_groq import ChatGroq

from dotenv import load_dotenv
import os
load_dotenv()

if os.environ['GROQ_API_KEY']:
    print("API Key is set.")
else:
    raise ValueError("API Key is not set.")

API Key is set.


In [7]:
llm = ChatGroq(model="llama-3.1-8b-instant")

In [1]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START,END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool

In [2]:
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

In [3]:
@tool
def add_numbers(a: int, b: int) -> int:
    """Adds two numbers together."""
    return a + b

In [8]:
tools = [add_numbers]
llm_with_tools = llm.bind_tools(tools)

In [9]:
def agent_node(state: AgentState) -> dict:
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}


def tool_node(state: AgentState) -> dict:
    last_message = state["messages"][-1]
    
    results = []
    for tool_call in last_message.tool_calls:
        if tool_call["name"] == "add_numbers":
            result = add_numbers.invoke(tool_call["args"])
            results.append(
                ToolMessage(
                    content=str(result),
                    tool_call_id=tool_call["id"]
                )
            )
    
    return {"messages": results}

In [10]:
def route(state: AgentState) -> str:
    last_message = state["messages"][-1]
    
    if last_message.tool_calls:
        return "tool_node"
    return "end"

In [13]:

graph = StateGraph(AgentState)


graph.add_node("agent_node", agent_node)
graph.add_node("tool_node", tool_node)


graph.add_edge(START,"agent_node")

graph.add_conditional_edges(
    "agent_node",
    route,
    {
        "tool_node": "tool_node",
        "end": END
    }
)

graph.add_edge("tool_node", "agent_node")


compiled_graph = graph.compile()

In [14]:
response = compiled_graph.invoke({
    "messages": [HumanMessage(content="What is 10 + 25?")]
})

for msg in response["messages"]:
    print(type(msg).__name__, ":", msg.content)

HumanMessage : What is 10 + 25?
AIMessage : 
ToolMessage : 35
AIMessage : The result of the function call is 35.
